# ESZG517 — Lab Session L5
## Anomaly Detection on Smart Home Sensor Data
### Using Isolation Forest

**Instructions:**
1. Upload `l5_sensor_data.csv` using the Files panel on the left (folder icon)
2. Run each cell in order by clicking the play button ▶ or pressing Shift+Enter
3. Take a screenshot of the final graph — save it as `l5_anomaly_graph.png`

---

In [ ]:
# ============================================================
# CELL 1 — Install and Import Libraries
# ============================================================
# scikit-learn contains the Isolation Forest algorithm
# pandas handles our CSV data
# matplotlib draws the graph

!pip install scikit-learn pandas matplotlib --quiet

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import datetime

print('Libraries loaded successfully.')

In [ ]:
# ============================================================
# CELL 2 — Load and Preview the Data
# ============================================================
# Read the CSV file you uploaded
# Convert the Unix timestamp to a readable datetime

df = pd.read_csv('l5_sensor_data.csv')
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')

print(f'Loaded {len(df)} sensor readings')
print(f'Columns: {list(df.columns)}')
print()
print('First 5 rows:')
df[['datetime','indoor_temp','ambient_lux','smoke_ppm','sound_db']].head()

In [ ]:
# ============================================================
# CELL 3 — Run Isolation Forest Anomaly Detection
# ============================================================
# Isolation Forest is an unsupervised ML algorithm.
# It does NOT need labelled data — it finds anomalies on its own
# by isolating points that are easy to separate from the rest.
#
# contamination=0.05 means we expect roughly 5% of readings
# to be anomalies. Adjust this if needed.

# Select the sensor features to analyse
features = ['indoor_temp', 'ambient_lux', 'smoke_ppm', 'sound_db']
X = df[features]

# Normalise the data so all sensors are on the same scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train the Isolation Forest model
model = IsolationForest(contamination=0.05, random_state=42)
df['anomaly_score'] = model.fit_predict(X_scaled)

# -1 = anomaly, 1 = normal
df['is_detected'] = df['anomaly_score'].apply(lambda x: 1 if x == -1 else 0)

total     = len(df)
anomalies = df['is_detected'].sum()
print(f'Total readings : {total}')
print(f'Anomalies found: {anomalies}')
print(f'Anomaly rate   : {anomalies/total*100:.1f}%')

In [ ]:
# ============================================================
# CELL 4 — Plot the Results
# ============================================================
# This graph shows all four sensors over time.
# Red dots mark readings flagged as anomalies by the model.
# Take a screenshot of this graph for your submission.

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
fig.suptitle('ESZG517 L5 — Smart Home Anomaly Detection\nIsolation Forest Results', 
             fontsize=14, fontweight='bold')

sensors = [
    ('indoor_temp',  'Indoor Temperature (°C)', '#2196F3'),
    ('ambient_lux',  'Ambient Light (lux)',      '#FF9800'),
    ('smoke_ppm',    'Smoke (ppm)',               '#9C27B0'),
    ('sound_db',     'Sound (dB)',                '#4CAF50'),
]

for ax, (col, label, color) in zip(axes, sensors):
    # Normal readings
    normal = df[df['is_detected'] == 0]
    ax.plot(normal['datetime'], normal[col], 
            color=color, linewidth=1, label='Normal')
    
    # Anomaly readings
    anomaly = df[df['is_detected'] == 1]
    ax.scatter(anomaly['datetime'], anomaly[col],
               color='red', s=80, zorder=5, label='Anomaly', marker='o')
    
    ax.set_ylabel(label, fontsize=9)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
plt.xticks(rotation=30)
plt.xlabel('Time')
plt.tight_layout()
plt.savefig('l5_anomaly_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graph saved as l5_anomaly_graph.png')
print('Right-click the file in the Files panel on the left to download it.')